In [25]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()



True

In [26]:
model=ChatOpenAI(model="gpt-5-mini")

In [27]:
class BlogState(TypedDict):
    

    title: str
    outline: str
    content: str

In [28]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state



def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [29]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [30]:
intial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(intial_state)

print(final_state)

{'title': 'Rise of AI in India', 'outline': 'Here’s a detailed, ready-to-use blog outline on “Rise of AI in India.” It includes structure, key points to cover in each section, suggested visuals, SEO elements, and content-length guidance so you can draft a polished long-form post.\n\nSuggested blog length: 1,800–2,500 words (adjustable)\nTone: Informative, forward-looking, evidence-based\nAudience: Policy readers, tech professionals, business leaders, informed general readers\n\nTitle ideas\n- The Rise of AI in India: Opportunities, Challenges and What’s Next\n- How AI Is Transforming India: A Sector-by-Sector Look\n- India’s AI Awakening: Policy, Startups, and Real-World Impact\n\nSEO meta description (150–160 chars)\nA concise look at India’s AI boom—policy push, startups, sector use-cases, risks and the roadmap for scaling responsible AI across the economy.\n\nPrimary keywords\nAI in India, artificial intelligence India, AI startups India, India AI policy, AI use cases India\n\nOutli